# Calibration

Trying out new calibration method.
Using xy files generated from mzXML sum spectra (VisuCon samples), containing
data for IgG, IgA and IgM.

## Data preparation


In [1]:
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make paths independent of whether Jupyter starts in the repository root
# or in the tests directory containing this notebook.
project_root = Path.cwd()
if not (project_root / "sweet_suite").is_dir():
    project_root = project_root.parent
if not (project_root / "sweet_suite").is_dir():
    raise FileNotFoundError("Could not locate the SweetSuite project root")
sys.path.insert(0, str(project_root))

from sweet_suite.mass_spectrometry.mass_spectrum import MassSpectrum

In [ ]:
xy_folder = project_root / "tests" / "xy_uncalibrated_sumspectra"

analytes_ref_path = project_root / "tests" / "analytes_ref.xlsx"
analytes_ref = pd.read_excel(analytes_ref_path)
ref_calibrants = analytes_ref[analytes_ref["calibrant"]]

BACKGROUND_MASS_WINDOW = 10.0
CALIBRATION_MASS_WINDOW = 0.2
CALIBRANT_SN_CUTOFF = 27.0

In [3]:
SUM_SPECTRUM_PATTERN = re.compile(
    r"^SumSpectrum_(?P<time>[^_]+)_(?P<time_window>[^_]+)_(?P<file_raw>.+)$"
)

def parse_sum_spectrum_path(path):
    match = SUM_SPECTRUM_PATTERN.fullmatch(Path(path).stem)
    return (
        float(match["time"]),
        float(match["time_window"]),
        match["file_raw"],
    )

def read_xy_file(path):
    return np.loadtxt(path)

In [4]:
# Check number of sum spectra created per mzXML.
xy_paths_by_file = {}
for path in xy_folder.glob("*.xy"):
    time, time_window, file_raw = parse_sum_spectrum_path(path)
    xy_paths_by_file.setdefault(file_raw, []).append(
        (time, time_window, path)
    )

for xy_paths in xy_paths_by_file.values():
    xy_paths.sort(key=lambda item: (item[0], item[1]))

group_summary = pd.DataFrame(
    {
        "raw_file": file_raw,
        "plate_position": file_raw.rsplit("-", 1)[-1],
        "sum_spectra": len(xy_paths),
    }
    for file_raw, xy_paths in sorted(xy_paths_by_file.items())
)

group_summary

,raw_file,plate_position,sum_spectra
0,aligned_2026GLY00034-PL01_B09,PL01_B09,12
1,aligned_2026GLY00034-PL01_D06,PL01_D06,12
2,aligned_2026GLY00034-PL01_E01,PL01_E01,12
3,aligned_2026GLY00034-PL01_G12,PL01_G12,12
4,aligned_2026GLY00034-PL02_B07,PL02_B07,12
5,aligned_2026GLY00034-PL02_E01,PL02_E01,12
6,aligned_2026GLY00034-PL02_F09,PL02_F09,12
7,aligned_2026GLY00034-PL02_G04,PL02_G04,12
8,aligned_2026GLY00034-PL02_H11,PL02_H11,12


We can now replicate part of the batch process.

In [5]:
mass_spectra = {}

# Loop over raw file (indicating mzXML file) and corresponding xy files
for file_raw, xy_paths in sorted(xy_paths_by_file.items()):

    file_mass_spectra = []  # To collect mass spectrum instances for this mzXML

    # Loop over xy sum spectra
    for time, time_window, path in xy_paths:

        # Select calibrants in this retention time range
        calibrants_df = (
            ref_calibrants[
                (ref_calibrants["time"] == time)
                & (ref_calibrants["time_window"] == time_window)
            ]
            .assign(
                charge=lambda frame: (
                    frame["peak"].str.split("_").str[1].astype(int)
                )
            )
            [["mz", "charge", "mz_window"]]
        )

        # Create a MassSpectrum instance and add to list.
        file_mass_spectra.append(
            MassSpectrum(
                name=path.stem,
                file_raw=file_raw,
                data_uncalibrated=read_xy_file(path),
                background_mass_window=BACKGROUND_MASS_WINDOW,
                calibration_mass_window=CALIBRATION_MASS_WINDOW,
                calibrants_df=calibrants_df,
                time=time,
                time_window=time_window,
            )
        )

    # Add list of Mass Spectra for this mzXML file to the dictionary.
    mass_spectra[file_raw] = file_mass_spectra


# Inspect number of spectra per mzXML as a sanity check.
{file_raw: len(spectra) for file_raw, spectra in mass_spectra.items()}

{'aligned_2026GLY00034-PL01_B09': 12,
 'aligned_2026GLY00034-PL01_D06': 12,
 'aligned_2026GLY00034-PL01_E01': 12,
 'aligned_2026GLY00034-PL01_G12': 12,
 'aligned_2026GLY00034-PL02_B07': 12,
 'aligned_2026GLY00034-PL02_E01': 12,
 'aligned_2026GLY00034-PL02_F09': 12,
 'aligned_2026GLY00034-PL02_G04': 12,
 'aligned_2026GLY00034-PL02_H11': 12}

In [13]:
# Collect all calibrants per mzXML file.
calibrants_dict = {}

for file_raw, file_mass_spectra in mass_spectra.items():
    
    file_calibrants = []

    for ms in file_mass_spectra:
        for cal in ms.calibrants:
            if cal.signal_to_noise > CALIBRANT_SN_CUTOFF:
                file_calibrants.append(
                    {
                        "time": cal.time,
                        "signal_to_noise": cal.signal_to_noise,
                        "mz_exact": cal.mz_exact,
                        "mz_observed": cal.mz_observed,
                    }
                )

    calibrants_dict[file_raw] = file_calibrants

pd.DataFrame(
    {"raw_file": file_raw, "calibrants": len(values)}
    for file_raw, values in calibrants_dict.items()
)

,raw_file,calibrants
0,aligned_2026GLY00034-PL01_B09,34
1,aligned_2026GLY00034-PL01_D06,49
2,aligned_2026GLY00034-PL01_E01,70
3,aligned_2026GLY00034-PL01_G12,53
4,aligned_2026GLY00034-PL02_B07,58
5,aligned_2026GLY00034-PL02_E01,75
6,aligned_2026GLY00034-PL02_F09,53
7,aligned_2026GLY00034-PL02_G04,65
8,aligned_2026GLY00034-PL02_H11,51


## Calibration methods
We now have a dictionary called `calibrants_dict`, which has a list
of calibrants for each mzXML file (this is the `calibrants` list in 
`batch_worker.py` line 790).

We'll start with just one file to try out some calibration methods.

In [ ]:
# Calibrants list of the first raw file.
# It is a list of dictionaries, each dictionary containing:
# - time
# - signal_to_noise
# - mz_exact
# - mz_observed
calibrants = calibrants_dict["aligned_2026GLY00034-PL01_B09"]
print(calibrants)

[{'time': 41.0, 'signal_to_noise': np.float64(70.17431773695404), 'mz_exact': 715.5480595721701, 'mz_observed': 715.54462542}, {'time': 41.0, 'signal_to_noise': np.float64(172.9530261824815), 'mz_exact': 1050.760131112394, 'mz_observed': 1050.7405458819064}, {'time': 41.0, 'signal_to_noise': np.float64(558.7584107222599), 'mz_exact': 788.3219175148328, 'mz_observed': 788.3113066686548}, {'time': 41.0, 'signal_to_noise': np.float64(46.62591866087587), 'mz_exact': 952.378828138957, 'mz_observed': 952.36237803713}, {'time': 87.5, 'signal_to_noise': np.float64(477.03820484772984), 'mz_exact': 1318.028118074156, 'mz_observed': 1318.0215757341016}, {'time': 87.5, 'signal_to_noise': np.float64(147.8933853039233), 'mz_exact': 879.0211709692686, 'mz_observed': 879.019067832963}, {'time': 87.5, 'signal_to_noise': np.float64(1746.4526292297594), 'mz_exact': 1399.05453865521, 'mz_observed': 1399.0431379502108}, {'time': 87.5, 'signal_to_noise': np.float64(911.6503465959114), 'mz_exact': 933.038784